# Drive 연결

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# 문제 1

[문제 1] - 한 글로벌 IT 기업의 HR 부서는 최근 이직률 증가로 인한 인력 손실과 채용 비용 부담이 커지고 있습니다. 현재 연간 이직률은 약 16%로 산업 평균(12%)보다 높으며, 특히 핵심 인재의 이직이 프로젝트 지연과 생산성 저하로 이어지고 있습니다. HR 팀은 데이터 기반 의사결정을 통해 이직 위험이 높은 직원을 사전에 파악하고 효과적인 인재 유지 전략을 수립하고자 합니다. 1,470명의 직원 데이터를 활용하여 이직 예측 모델을 개발하고, 이직에 영향을 미치는 주요 요인을 분석하는 것이 목표입니다.

[데이터]
- 파일명: 10_3_1.csv
- 출처: IBM HR Analytics Employee Attrition & Performance Dataset
- 다운로드 : https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset
- 데이터 크기: 1,470명의 직원 데이터, 35개 변수

[Data Description]
※ `10_3_1.csv`는 IBM에서 생성한 가상의 인사 데이터셋으로, 직원의 이직(Attrition) 여부와 관련된 다양한 개인적·직무적 특성을 포함합니다. HR 분석에서 직원 이탈 예측 및 주요 영향 요인 파악을 위해 설계되었으며, 전 세계적으로 HR 분석 교육 및 연구에 널리 사용되는 표준 데이터셋입니다.

[컬럼 설명]
- Age: 나이 (18~60세)
- Gender: 성별 - Male, Female
- Education: 교육 수준 (1~5) - 1: Below College, 2: College, 3: Bachelor, 4: Master, 5: Doctor
- MonthlyIncome: 월 급여 (1,009~19,999 USD)
- OverTime: 초과 근무 여부 - Yes, No
- DistanceFromHome: 집에서 회사까지 거리 (1~29 miles)
- YearsAtCompany: 현 회사 근무 연수 (0~40년)
- Attrition: 이직 여부 (종속변수) - Yes(이직), No(잔류)

[문항]
(1) 모든 독립변수를 사용하여 이직 여부를 예측하는 로지스틱 회귀모형을 적합하고, 유의한 변수(유의확률 0.05 미만)의 회귀계수를 소수점 셋째 자리까지 반올림하여 제출하시오. (단, 절편 제외)

(2) 위 (1)번에서 적합한 모형에서 Education이 1단계 증가할 때의 이직 오즈비(Odds Ratio)를 소수점 셋째 자리까지 반올림하여 제출하시오.

(3) 다음 조건을 가진 직원의 이직 확률을 위 (1)번 모형으로 예측하여 소수점 셋째 자리까지 반올림하여 제출하시오
다음 조건(아래 컬럼 모두 포함)에 대해 이직 확률을 예측하시오.
- Age: 40세
- Gender: Male
- Education: 3 (Bachelor)
- MonthlyIncome: 3,000
- OverTime: Yes
- DistanceFromHome: 10
- YearsAtCompany: 2





In [19]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1) 데이터 로드
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/Udemy/빅데이터분석기사_파이썬/작업형_제3유형/10회/data/10_3_1.csv"
df = pd.read_csv(DATA_PATH)

# print(df.head())

# 타깃 이진화 (Yes=1, No=0)
df["Attrition_bin"] = (df["Attrition"].astype(str).str.strip() == "Yes").astype(int)

# print(df.head())

# 숫자형 변환
num_cols = ["Age", "Education", "MonthlyIncome", "DistanceFromHome", "YearsAtCompany"]
for c in num_cols:
  df[c] = pd.to_numeric(df[c], errors="coerce")

# print(df.isna().sum(), "\n")

# (1) 로지스틱 회귀 적합
# - 범주형: C(Gender), C(OverTime)
formula = (
    "Attrition_bin ~ Age + Education + MonthlyIncome + DistanceFromHome + YearsAtCompany"
    " + C(Gender) + C(OverTime)"
)

# - 모형 적합
# - disp=False는 학습 과정 출력(반복 로그) 숨기기
# - maxiter=200, 최적화 반복 횟수 늘려 수렴 실패 줄이기 위한 안전장치
model = smf.logit(formula=formula, data=df).fit(disp=False, maxiter=200)
# print(model.summary())

# 유의한 변수 (p < 0.05) 계수 출력
params = model.params.drop(["Intercept"])
pvals = model.pvalues.drop(["Intercept"])
# print(pvals)

sig_mask = pvals < 0.05
sig_params = params[sig_mask].sort_values()
# print(sig_params)

# (2) Education 오즈비 (OR) = exp(beta_Education)
beta_edu = float(model.params["Education"])
or_edu = float(np.exp(beta_edu))
print("(2) Education 1단계 증가 시 오즈비(OR) =", round(or_edu, 3))

# (3) 조건 직원 이직 확률

new = pd.DataFrame({
        "Age": [40],
        "Gender": ["Male"],
        "Education": [3],
        "MonthlyIncome": [3000],
        "OverTime": ["Yes"],
        "DistanceFromHome": [10],
        "YearsAtCompany": [2]
})

pred_prob = float(model.predict(new)[0])
print("(3) 조건 직원 이직 확률=", round(pred_prob, 3))

(2) Education 1단계 증가 시 오즈비(OR) = 1.033
(3) 조건 직원 이직 확률= 0.388


# 문제 2

[문제 2] - 한 부동산 컨설팅 회사는 주택 가격 예측 시스템을 개발하여 고객에게 적정 매매가를 제시하고자 합니다. 아이오와주 에임스(Ames) 지역의 과거 주택 거래 데이터를 활용하여 주택의 물리적 특성과 품질이 가격에 미치는 영향을 분석하고, 새로운 매물의 예상 가격을 산정하는 모델을 구축하는 것이 목표입니다.당신은 데이터 분석가로서 다중선형회귀 모델을 활용하여 주택 가격 예측 시스템을 개발하게 되었습니다. 제공된 데이터를 활용하여 아래 문제를 풀이하시오.

[데이터]
- 파일명: 10_3_2.csv
- 출처: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data

[Data Description]
※ `10_3_2.csv`는 아이오와주 에임스 지역의 2006-2010년 주택 거래 데이터로, 주택의 물리적 특성, 품질, 위치 등 다양한 속성과 실제 판매 가격을 포함합니다. 부동산 가격 예측 및 회귀 분석 연구에 널리 사용되는 표준 데이터셋입니다.


[컬럼 설명]
- OverallQual: 전반적인 자재 및 마감 품질 (1~10점) - 10: Very Excellent, 5: Average, 1: Very Poor
- OverallCond: 전반적인 주택 상태 (1~10점) - 10: Very Excellent, 5: Average, 1: Very Poor
- GrLivArea: 지상층 거주 면적 (square feet)
- TotalBsmtSF: 지하실 총 면적 (square feet)
- 1stFlrSF: 1층 면적 (square feet)
- GarageArea: 차고 면적 (square feet)
- FullBath: 지상층 전체 욕실 개수
- BedroomAbvGr: 지상층 침실 개수
- TotRmsAbvGrd: 지상층 총 방 개수 (욕실 제외)
- YearBuilt: 건축 연도 (1872~2010)
- SalePrice: 주택 판매 가격 (USD)

[문항]
(1) SalePrice를 종속변수로 하고, 다음 10개 독립 변수를 독립변수로 하는 다중선형회귀모형을 적합하시오. 유의수준 0.05에서 통계적으로 유의한 변수들의 회귀계수 합을 소수점 둘째 자리까지 반올림하여 제출하시오. (단, 절편은 제외)

(2) 위 (1)번에서 유의한 변수(p < 0.05)만을 독립변수로 사용하여 다중선형회귀모형을 다시 적합하시오. 이 모형의 **결정계수(R²)**를 소수점 셋째 자리까지 반올림하여 제출하시오.

(3) 다음 조건을 가진 주택의 가격을 위 (2)번 모형(유의한 변수만 사용)으로 예측하여 소수점 둘째 자리까지 반올림하여 제출하시오
- (2)번에서 선택된 유의한 변수에 해당하는 값만 사용하여 예측하세요. 유의하지 않은 변수의 조건은 무시하세요.
- 예시
    + OverallQual = 7
    + OverallCond = 5
    + GrLivArea = 1,500
    + TotalBsmtSF = 1,000
    + 1stFlrSF = 1,000
    + GarageArea = 500
    + FullBath = 2
    + BedroomAbvGr = 3
    + TotRmsAbvGrd = 7
    + YearBuilt = 2000




In [36]:
import pandas as pd
import statsmodels.api as sm

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/Udemy/빅데이터분석기사_파이썬/작업형_제3유형/10회/data/"

# 1) 데이터 로드
df = pd.read_csv(DATA_PATH + "10_3_2.csv")
# print(df.head())

# - 문제에서 지정한 10개 독립변수
FEATURES = ["OverallQual", "OverallCond", "GrLivArea", "TotalBsmtSF",     "1stFlrSF", "GarageArea", "FullBath", "BedroomAbvGr", "TotRmsAbvGrd", "YearBuilt",
]

TARGET = "SalePrice"

# 숫자형 반환
for c in FEATURES + [TARGET]:
  df[c] = pd.to_numeric(df[c], errors="coerce")

use_df = df[FEATURES + [TARGET]].dropna().copy()
print("[사용 데이터] shape:", use_df.shape)

# (1) 10개 변수로 OLS 적합 -> 유의 변수 계수 합 (절편 제외)
X = sm.add_constant(use_df[FEATURES])
y = use_df[TARGET].astype(float)

model1 = sm.OLS(y, X).fit()
# print("[모형1 요약]")
# print(model1.summary())

params1 = model1.params.drop("const")
pvals1 = model1.pvalues.drop("const")

sig_vars = pvals1[pvals1 < 0.05].index.tolist()
sig_sum = float(params1.loc[sig_vars].sum()) if len(sig_vars) > 0 else 0.0
print("(1) 유의 변수 (p<0.05) 회귀계수 합(절편 제외)=", round(sig_sum, 2))
print("- 유의 변수 목록:", sig_vars)

# (2) 유의 변수만으로 재적합
X2 = sm.add_constant(use_df[sig_vars])
model2 = sm.OLS(y, X2).fit()
# print(model2.summary())

r2 = round(float(model2.rsquared), 3)
print("(2) 유의 변수만 사용한 모형의 R2=", r2)

# (3), (2)에서 만든 모형으로 예측
# - 유의하지 않은 변수 조건은 생략

new = {
    "OverallQual": 7,
    "OverallCond": 5,
    "GrLivArea": 1500,
    "TotalBsmtSF": 1000,
    "1stFlrSF": 1000,
    "GarageArea": 500,
    # "FullBath": 2,
    "BedroomAbvGr": 3,
    "TotRmsAbvGrd": 7,
    "YearBuilt": 2000,
}

new_X = pd.DataFrame(new, sig_vars)
new_X = sm.add_constant(new_X, has_constant="add")
pred = float(model2.predict(new_X)[0])
print("(3) 예측 SalePrice =", round(pred, 2))

[사용 데이터] shape: (1460, 11)
(1) 유의 변수 (p<0.05) 회귀계수 합(절편 제외)= 19501.27
- 유의 변수 목록: ['OverallQual', 'OverallCond', 'GrLivArea', 'TotalBsmtSF', '1stFlrSF', 'GarageArea', 'BedroomAbvGr', 'TotRmsAbvGrd', 'YearBuilt']
(2) 유의 변수만 사용한 모형의 R2= 0.781
(3) 예측 SalePrice = 204844.16


/tmp/ipython-input-3646115959.py:65: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pred = float(model2.predict(new_X)[0])
